# 12a - Mean-of-folds vs Pooled-OOF (canonical results aggregator)

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


Recomputes every experiment's honest cross-city metrics from committed OOF, replacing all Claude-computed
mean-folds (those were PROVISIONAL). CV = **leave-one-city-out (21 folds)**, so **mean-of-folds = mean of per-city
AUCs**. Reads OOF from BOTH roots. Uses only held-out rows (`is_final == False`).

Outputs (to `RESULTS_ROOT/nb12/`):
- `nb12a_meanfolds_vs_pooled.csv` - per experiment: pooled, pooled-no-Mariupol, mean-folds, std, gap
- `nb12a_per_city_auc_matrix.csv` - experiment x city AUC matrix (input for 12b inferential stats + 12c)

Run in WSL/conda/JupyterLab. Writes results only; trains nothing.

In [1]:
import sys, re, collections
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score

_known = [Path("/content/drive_f/masterthesis/notebooks"),
          Path("/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks"),
          Path(r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks")]
_nb = next((c for c in list(Path.cwd().parents) + _known if (c / "global_setup.py").exists()), None)
if _nb is None:
    raise RuntimeError("global_setup.py not found - run from inside the notebooks tree")
if str(_nb) not in sys.path:
    sys.path.insert(0, str(_nb))
import global_setup as gs

DRIVE_ROOT   = Path(gs.DRIVE_ROOT)
RESULTS_ROOT = Path(gs.RESULTS_ROOT)
OUT_DIR = RESULTS_ROOT / "nb12"; OUT_DIR.mkdir(parents=True, exist_ok=True)

OOF_ROOTS = []
for attr in ["RESULTS_ROOT","OUTPUT_ROOT","DATA_OUTPUTS"]:
    v = getattr(gs, attr, None)
    if v: OOF_ROOTS.append(Path(v))
OOF_ROOTS += [DRIVE_ROOT / "data" / "outputs", RESULTS_ROOT]
seen=set(); OOF_ROOTS=[r for r in OOF_ROOTS if r.exists() and (r not in seen and not seen.add(r))]
print("OOF roots:", *[str(r) for r in OOF_ROOTS], sep="\n  ")
print("output   :", OUT_DIR)

MARIUPOL = "Mariupol"

def safe_auc(y, s):
    y = np.asarray(y); s = np.asarray(s)
    if len(np.unique(y)) < 2: return np.nan
    try: return float(roc_auc_score(y, s))
    except Exception: return np.nan

/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


BDA GLOBAL SETUP
Started: 2026-06-23 06:34:11
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------
  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: False
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


/content/drive_f/masterthesis/notebooks/global_setup.py:564: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     934.7/7452.0 GB (6517.3 GB free)
  GDrive (F:)     1405.1/3726.0 GB (2320.9 GB free)
  Local data      11565.2/14901.9 GB (3336.7 GB free)
  Data stack      1405.1/3726.0 GB (2320.9 GB free)
  WSL ext4        69.0/1006.9 GB (886.6 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_

## Discover OOF and deduplicate (keep latest run per experiment)

Each experiment can have several OOF files (reruns). Dedup by the filename signature (experiment + model + variant),
keeping the latest timestamp. Filename pattern: `oof_<sig>__<YYYYMMDD>_<HHMMSS>_<hash>.parquet`.

In [2]:
def sig_from_name(name):
    s = re.sub(r"\.parquet$", "", name)
    s = re.sub(r"^oof_", "", s)
    s = re.sub(r"__\d{8}_\d{6}_[0-9a-fA-F]+$", "", s)  # strip trailing __ts_hash
    return s
def ts_from_name(name):
    m = re.search(r"(\d{8})_(\d{6})", name)
    return m.group(0) if m else ""

paths = []
for root in OOF_ROOTS:
    paths += list(root.rglob("oof_*.parquet"))
paths = sorted(set(paths))
print(f"found {len(paths)} oof files across {len(OOF_ROOTS)} roots")

inv = pd.DataFrame({"path": paths})
inv["name"] = inv["path"].map(lambda p: p.name)
inv["sig"]  = inv["name"].map(sig_from_name)
inv["ts"]   = inv["name"].map(ts_from_name)
dedup = inv.sort_values("ts").drop_duplicates("sig", keep="last").reset_index(drop=True)
print(f"after dedup: {len(dedup)} experiments")
print(dedup[["sig","ts"]].head(12).to_string(index=False))

found 1789 oof files across 2 roots
after dedup: 489 experiments
                           sig ts
               R0a_eq3_guarded   
              R0a_eq3_post_max   
      R0_dietrich28_groupkfold   
    R0b_roll13_assessment_only   
R0b_roll13_baseline+assessment   
     R0b_roll2_assessment_only   
 R0b_roll2_baseline+assessment   
     R0b_roll3_assessment_only   
 R0b_roll3_baseline+assessment   
     R0b_roll5_assessment_only   
 R0b_roll5_baseline+assessment   
     R0b_roll7_assessment_only   


## Compute metrics per experiment

For each deduped experiment: keep held-out rows (`is_final == False`), then compute pooled AUC, pooled-without-
Mariupol, per-city AUCs (= per-fold), and mean/std of per-city AUCs. Cities with a single class in OOF are skipped
for the per-city AUC (reported via `n_cities_scored`).

In [3]:
rows = []
pca_matrix = {}   # experiment -> {city: auc}

for _, r in dedup.iterrows():
    p = r["path"]; sig = r["sig"]
    try:
        avail = [f.name for f in pq.ParquetFile(p).schema_arrow]
    except Exception:
        continue
    use = [c for c in ["city","fold_id","y_true","y_proba","is_final"] if c in avail]
    if "y_proba" not in use or "y_true" not in use or "city" not in use:
        continue
    d = pd.read_parquet(p, columns=use)
    if "is_final" in d.columns:
        d = d[~d["is_final"].astype(bool)]
    if d.empty:
        continue

    pooled  = safe_auc(d["y_true"], d["y_proba"])
    nomar   = d[d["city"] != MARIUPOL]
    pooled_nm = safe_auc(nomar["y_true"], nomar["y_proba"]) if len(nomar) else np.nan

    pca = {c: safe_auc(g["y_true"], g["y_proba"]) for c, g in d.groupby("city")}
    pca_matrix[sig] = pca
    vals = np.array([v for v in pca.values() if not np.isnan(v)])
    mean_f = float(np.mean(vals)) if len(vals) else np.nan
    std_f  = float(np.std(vals))  if len(vals) else np.nan

    rows.append({
        "experiment": sig,
        "n_rows": int(len(d)),
        "n_cities_total": int(d["city"].nunique()),
        "n_cities_scored": int(len(vals)),
        "pooled_auc": pooled,
        "pooled_auc_no_mariupol": pooled_nm,
        "mean_folds_auc": mean_f,
        "std_folds_auc": std_f,
        "gap_pooled_minus_folds": (pooled - mean_f) if (mean_f==mean_f and pooled==pooled) else np.nan,
    })

res = pd.DataFrame(rows).sort_values("mean_folds_auc", ascending=False, na_position="last").reset_index(drop=True)
res.to_csv(OUT_DIR / "nb12a_meanfolds_vs_pooled.csv", index=False)

# per-city AUC matrix (experiment x city)
all_cities = sorted({c for d in pca_matrix.values() for c in d})
mat = pd.DataFrame(index=sorted(pca_matrix), columns=all_cities, dtype=float)
for sig, d in pca_matrix.items():
    for c, v in d.items():
        mat.loc[sig, c] = v
mat.to_csv(OUT_DIR / "nb12a_per_city_auc_matrix.csv")

print(f"{len(res)} experiments scored. Saved:")
print("  ", (OUT_DIR / 'nb12a_meanfolds_vs_pooled.csv'))
print("  ", (OUT_DIR / 'nb12a_per_city_auc_matrix.csv'))

458 experiments scored. Saved:
   /content/drive_f/masterthesis/results/nb12/nb12a_meanfolds_vs_pooled.csv
   /content/drive_f/masterthesis/results/nb12/nb12a_per_city_auc_matrix.csv


## Results table + interpretation

Report BOTH mean-folds (honest cross-city) and pooled (Mariupol-dominated). The gap is driven by Mariupol holding
41.7% of all positive labels; the pooled-without-Mariupol column isolates that effect.

In [4]:
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)
cols = ["experiment","n_cities_scored","mean_folds_auc","std_folds_auc",
        "pooled_auc","pooled_auc_no_mariupol","gap_pooled_minus_folds"]
print("TOP 20 by mean-of-folds AUC:\n")
print(res[cols].head(20).to_string(index=False))

print("\n--- summary ---")
if len(res):
    best = res.iloc[0]
    print(f"best mean-folds : {best['experiment']}  {best['mean_folds_auc']:.4f} +/- {best['std_folds_auc']:.4f} "
          f"(n_cities={best['n_cities_scored']})")
    print(f"  its pooled    : {best['pooled_auc']:.4f}  | pooled-no-Mariupol: {best['pooled_auc_no_mariupol']:.4f}")
    med_gap = res['gap_pooled_minus_folds'].median(skipna=True)
    print(f"median pooled-minus-folds gap across experiments: {med_gap:+.4f}")
print("\nNOTE: cite mean-folds as the honest cross-city estimate (= expected unseen-city performance).")
print("All numbers now reproducible from committed OOF (supersedes Claude-computed mean-folds).")

TOP 20 by mean-of-folds AUC:

                          experiment  n_cities_scored  mean_folds_auc  std_folds_auc  pooled_auc  pooled_auc_no_mariupol  gap_pooled_minus_folds
               NB08b_BDA_RGB_XGBoost               21        0.748478       0.068605    0.661497                0.699958               -0.086981
              NB08b_BDA_RGB_AdaBoost               21        0.748367       0.069239    0.712075                0.732714               -0.036292
NB08b_BDA_COH_DROP+RGB+CARD_AdaBoost               12        0.745562       0.071589    0.738145                0.762562               -0.007416
                NB08b_BDA_RGB_LogReg               21        0.744569       0.071467    0.730733                0.739839               -0.013835
     NB08b_BDA_COH_DROP+RGB_AdaBoost               12        0.744086       0.074172    0.730888                0.770847               -0.013197
            NB08b_BDA_RGB_ExtraTrees               21        0.743519       0.070003    0.750326    